# Modelagem em Grafos e Detecção de Comunidades Aplicadas à Base SAIC (Constituinte de 1988)

Este notebook reproduz integralmente a inspeção da base de dados higienizada (`SAIC.csv`), o pré-processamento de linguagem natural (`spaCy` — `pt_core_news_md`), a construção do grafo de coocorrência de palavras ponderado por **Positive Pointwise Mutual Information (PPMI)**, a detecção de comunidades pelo algoritmo de **Louvain** e a extração de métricas de centralidade e correspondência temática.

## 1. Instalação e Importação de Bibliotecas

In [ ]:
# Descomente a linha abaixo caso esteja executando no Google Colab:
# !pip install -q spacy scikit-learn networkx matplotlib pandas numpy && python -m spacy download pt_core_news_md

import pandas as pd
import numpy as np
import spacy
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

## 2. Carregamento e Auditoria da Base Tratada (`SAIC.csv`)

In [ ]:
CSV_PATH = 'SAIC.csv'
df = pd.read_csv(CSV_PATH, sep=';', low_memory=False)
print(f'Dimensões da base: {df.shape[0]} linhas x {df.shape[1]} colunas\n')

summary_rows = []
for col in df.columns:
    summary_rows.append({
        'Coluna': col,
        'N/A': int(df[col].isna().sum()),
        'Valores Únicos': int(df[col].nunique(dropna=True))
    })
df_summary = pd.DataFrame(summary_rows)
display(df_summary)

### 2.1. Distribuição de Frequência dos 24 Macrotemas Oficiais (`CATÁLOGO`)

In [ ]:
cat_counter = Counter()
for val in df['CATÁLOGO'].dropna():
    items = [x.strip() for x in str(val).split(',')]
    cat_counter.update(items)

df_cat = pd.DataFrame(cat_counter.most_common(), columns=['Tema (Catálogo)', 'Frequência'])
display(df_cat)

## 3. Pré-processamento Textual (Tokenização, Remoção de *Stopwords* e Lematização)

Desconsideram-se registros nulos e cópias exatas da coluna `TEXTO SUGESTÃO` para evitar inflar artificialmente a coocorrência.

In [ ]:
df_dedup = df.dropna(subset=['TEXTO SUGESTÃO']).drop_duplicates(subset=['TEXTO SUGESTÃO']).copy()
print(f'Total de cartas únicas válidas processadas (N): {len(df_dedup)}')

nlp = spacy.load('pt_core_news_md', disable=['ner', 'parser'])

processed_docs = []
texts = df_dedup['TEXTO SUGESTÃO'].astype(str).tolist()
for doc in nlp.pipe(texts, batch_size=1000):
    lemmas = [
        token.lemma_.lower().strip()
        for token in doc
        if not token.is_stop and not token.is_punct and not token.is_space and token.is_alpha
    ]
    processed_docs.append(' '.join(lemmas))

print('Pré-processamento concluído com sucesso!')

## 4. Construção da Matriz de Coocorrência, Ponderação por PPMI e Poda de Arestas

Abaixo são avaliadas as combinações de frequência mínima documental (`min_df` de 1,0% a 3,0%) e os percentis de corte da distribuição de PPMI (50, 75, 90 e 95), juntamente com a modularidade ($Q$) obtida pelo algoritmo de Louvain.

In [ ]:
def compute_ppmi_matrix(docs, min_df):
    vec = CountVectorizer(binary=True, min_df=min_df)
    A = vec.fit_transform(docs)
    vocab = np.array(vec.get_feature_names_out())
    n_docs, n_terms = A.shape
    
    C = (A.T @ A).toarray().astype(float)
    diag = np.diag(C).copy()
    np.fill_diagonal(C, 0.0)
    
    p_ij = C / n_docs
    p_i = diag / n_docs
    with np.errstate(divide='ignore', invalid='ignore'):
        pmi = np.log2(p_ij / np.outer(p_i, p_i))
    ppmi = np.maximum(0.0, np.nan_to_num(pmi, nan=0.0, posinf=0.0, neginf=0.0))
    return A, vocab, ppmi

exp_rows = []
for min_df in [0.01, 0.015, 0.02, 0.025, 0.03]:
    _, vocab, ppmi = compute_ppmi_matrix(processed_docs, min_df)
    n_terms = len(vocab)
    iu = np.triu_indices(n_terms, k=1)
    pos_vals = ppmi[iu][ppmi[iu] > 0]
    
    for perc in [50, 75, 90, 95]:
        corte = np.percentile(pos_vals, perc)
        G_tmp = nx.Graph()
        mask = ppmi[iu] >= corte
        for u_idx, v_idx, w in zip(iu[0][mask], iu[1][mask], ppmi[iu][mask]):
            G_tmp.add_edge(vocab[u_idx], vocab[v_idx], weight=float(w))
        
        comms_tmp = nx.community.louvain_communities(G_tmp, weight='weight', seed=42)
        Q = nx.community.modularity(G_tmp, comms_tmp, weight='weight')
        exp_rows.append({
            'min_df (%)': f'{min_df*100:.1f}%',
            'Percentil PPMI': perc,
            'Corte PPMI': round(float(corte), 2),
            'Arestas': G_tmp.number_of_edges(),
            'Nós': G_tmp.number_of_nodes(),
            'Densidade (%)': round(nx.density(G_tmp) * 100, 2),
            'Comp. Conexas': nx.number_connected_components(G_tmp),
            'Modularidade (Q)': round(float(Q), 4),
            'Nº Comunidades': len(comms_tmp)
        })

df_exp = pd.DataFrame(exp_rows)
display(df_exp)

## 5. Detecção de Comunidades (Louvain) e Métricas de Centralidade (`min_df = 2,0%`, Percentil 95)

In [ ]:
A_opt, vocab_opt, ppmi_opt = compute_ppmi_matrix(processed_docs, min_df=0.02)
n_terms = len(vocab_opt)
iu = np.triu_indices(n_terms, k=1)
pos_vals = ppmi_opt[iu][ppmi_opt[iu] > 0]
corte_95 = np.percentile(pos_vals, 95)

G = nx.Graph()
mask = ppmi_opt[iu] >= corte_95
for u_idx, v_idx, w in zip(iu[0][mask], iu[1][mask], ppmi_opt[iu][mask]):
    G.add_edge(vocab_opt[u_idx], vocab_opt[v_idx], weight=float(w))

comms = nx.community.louvain_communities(G, weight='weight', seed=42)
Q_opt = nx.community.modularity(G, comms, weight='weight')
print(f'Grafo ótimo: |V| = {G.number_of_nodes()}, |E| = {G.number_of_edges()}, Q = {Q_opt:.4f}, Comunidades = {len(comms)}\n')

cent_rows = []
for i, comm in enumerate(sorted(comms, key=len, reverse=True), 1):
    subG = G.subgraph(comm)
    top_deg = max(nx.degree_centrality(subG).items(), key=lambda x: x[1])[0]
    top_bet = max(nx.betweenness_centrality(subG).items(), key=lambda x: x[1])[0]
    top_clo = max(nx.closeness_centrality(subG).items(), key=lambda x: x[1])[0]
    cent_rows.append({
        'Comunidade #': i,
        'Nº Termos': len(comm),
        'Maior Grau': top_deg,
        'Maior Intermediação': top_bet,
        'Maior Proximidade': top_clo,
        'Amostra de Termos': ', '.join(sorted(list(comm))[:12])
    })

df_cent = pd.DataFrame(cent_rows)
display(df_cent)